In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")
import random
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
import lightgbm as lgb
import pickle

# 変数の設定

In [ ]:
class CFG:
    VER = 1
    AUTHOR = "suba"
    COMPETITION = "atmacup17"
    DATA_PATH = Path("/workspace/kaggle_llm_book/ch3/data/takaito/atmacup17")
    MODEL_NAME = "lightgbm"
    SEED = 42
    N_SPLIT = 3
    TARGET_COL = "Recommended IND"
    TARGET_COL_CLASS_NUM = 2
    METRIC = "auc"
    METRIC_MAXIMIZE_FLAG = True
    NUM_BOOST_ROUND = 10000
    EARLY_STOPPING_ROUND = 100
    VERBOSE = 250
    LGB_PARAMS= {
        "objective": "binary",
        "metric": "auc",
        "learning_rate": 0.01,
        "seed": SEED,
    }
    NGRAM_RANGE = (1, 2)
    LOWERCASE = True
    SUBLINEAR_TF = True
    MAX_DF = 0.8
    MIN_DF = 5
    SVD_DIM = 256
    PREFIX  = f"{AUTHOR}_{MODEL_NAME}_seed{SEED}_ver{VER}"

# 乱数の設定

In [ ]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
seed_everything(CFG.SEED)

# データの読み込みと前処理

In [ ]:
clothing_master_df = pd.read_csv(CFG.DATA_PATH / "clothing_master.csv")
train_df = pd.read_csv(CFG.DATA_PATH / "train.csv")
test_df = pd.read_csv(CFG.DATA_PATH / "test.csv")

In [ ]:
def make_text_column(df):
    df["text"] = df["Title"] + " " + df["Review Text"]
    return df

def preprocessing(df, clothing_master_df):
    df["Title"] = df["Title"].fillna("")
    df["Review Text"] = df["Review Text"].fillna("")
    df = df.merge(clothing_master_df, on="Clothing ID", how="left")
    df = make_text_column(df)
    return df

In [ ]:
train_df = preprocessing(train_df, clothing_master_df)
test_df = preprocessing(test_df, clothing_master_df)

In [ ]:
train_df.head()

# テキストから特徴量作成

In [ ]:
vectorizer = TfidfVectorizer(ngram_range=CFG.NGRAM_RANGE, 
                             lowercase=CFG.LOWERCASE, 
                             sublinear_tf=CFG.SUBLINEAR_TF, 
                             max_df=CFG.MAX_DF, min_df=CFG.MIN_DF
                            )

In [ ]:
vectorizer.fit(train_df["text"])

In [ ]:
print(vectorizer.get_feature_names_out())
print(len(vectorizer.get_feature_names_out()))

In [ ]:
train_tfidf_matrix = vectorizer.transform(train_df["text"])
test_tfidf_matrix = vectorizer.transform(test_df["text"])

In [ ]:
train_tfidf_matrix.toarray()

In [ ]:
svd = TruncatedSVD(n_components=CFG.SVD_DIM, random_state=CFG.SEED)

In [ ]:
svd.fit(train_tfidf_matrix)

In [ ]:
train_tfidf_svd_matrix = svd.transform(train_tfidf_matrix)
test_tfidf_svd_matrix = svd.transform(test_tfidf_matrix)

In [ ]:
print(train_tfidf_matrix.shape, train_tfidf_svd_matrix.shape)

# モデルの学習と推論

In [ ]:
def training_and_inference(train_df, test_df, train_matrix, test_matrix, ex_name="temp"):
    oof_predictions = np.zeros(len(train_df))
    test_predictions = np.zeros(len(test_df))
    kfold = StratifiedKFold(n_splits=CFG.N_SPLIT, shuffle=True, random_state=CFG.SEED)
    for fold, (train_index, valid_index) in enumerate(kfold.split(train_df, train_df[CFG.TARGET_COL])):
        print(f"training fold {fold+1}")
        x_train = train_matrix[train_index]
        y_train = train_df[CFG.TARGET_COL].iloc[train_index]
        x_valid = train_matrix[valid_index]
        y_valid = train_df[CFG.TARGET_COL].iloc[valid_index]
        model = lgb.train(
            params=CFG.LGB_PARAMS,
            train_set=lgb.Dataset(x_train, y_train),
            num_boost_round=CFG.NUM_BOOST_ROUND,
            valid_sets=[lgb.Dataset(x_train, y_train), 
                        lgb.Dataset(x_valid, y_valid)],
            callbacks=[lgb.early_stopping(stopping_rounds=CFG.EARLY_STOPPING_ROUND, verbose=CFG.VERBOSE),
                       lgb.log_evaluation(CFG.VERBOSE)]
        )
        # 検証セットの推論
        valid_pred = model.predict(x_valid)
        oof_predictions[valid_index] = valid_pred
        # テストセットの推論
        test_pred = model.predict(test_matrix)
        test_predictions += test_pred / CFG.N_SPLIT
        pickle.dump(model, open(f"{ex_name}_fold{fold + 1}.pkl", "wb"))
    # 交差検証のスコアの確認
    print("CV Score: ", roc_auc_score(train_df[CFG.TARGET_COL], oof_predictions))
    # submission.csv の作成
    test_df["target"] = test_predictions
    test_df[["target"]].to_csv(f"{ex_name}_submission.csv", index=False)

In [ ]:
# TF-IDF 行列を用いて学習 & 推論
training_and_inference(train_df, 
                       test_df, 
                       train_tfidf_matrix, 
                       test_tfidf_matrix, 
                       ex_name=f"TF-IDF_{CFG.PREFIX}")

In [ ]:
# SVD で次元圧縮した行列を用いて学習 & 推論
training_and_inference(train_df, 
                       test_df, 
                       train_tfidf_svd_matrix, 
                       test_tfidf_svd_matrix, 
                       ex_name=f"SVD_{CFG.PREFIX}")

In [ ]:
pickle.dump(vectorizer, open(f"vectorizer_{CFG.PREFIX}.pkl", "wb"))
pickle.dump(svd, open(f"svd_{CFG.PREFIX}.pkl", "wb"))